# 19 - ArTST v3 (End-to-End Diacritized ASR)

**Model:** `MBZUAI/artst_asr_v3`  
**Architecture:** SpeechT5-based (Transformers compatible)  
**Training:** Arabic ASR with diacritics support

**Key Feature:** This model outputs **diacritized Arabic text directly** from audio - no separate diacritization step needed!

**Tasks:**
1. Dev Set: ASR inference → Diacritized output + Metrics
2. Blind Test: ASR inference + Submission

In [1]:
!pip install -q transformers torch accelerate tqdm librosa soundfile

In [2]:
# Setup
import os, sys, json, re, torch, zipfile, subprocess
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import librosa
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

try:
    from config import DEV_AUDIO_DIR, DEV_INPUT, DEV_OUTPUT, TEST_DIR, TEST_AUDIO_DIR, OUTPUT_DIR, SUBMISSION_DIR
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found.")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.5 GB)


In [3]:
MODEL_KEY = 'artst_v3_diac'
ARTST_MODEL = 'MBZUAI/artst_asr_v3'

print(f"Loading {ARTST_MODEL}...")
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText

processor = SpeechT5Processor.from_pretrained(ARTST_MODEL)

# Load in float32 to avoid dtype mismatch with audio inputs
model = SpeechT5ForSpeechToText.from_pretrained(
    ARTST_MODEL,
    torch_dtype=torch.float32
).to(device)
model.eval()

print(f"ArTST v3 loaded! This model outputs diacritized text directly.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading MBZUAI/artst_asr_v3...


Loading weights:   0%|          | 0/369 [00:00<?, ?it/s]

SpeechT5ForSpeechToText LOAD REPORT from: MBZUAI/artst_asr_v3
Key                                                  | Status     |  | 
-----------------------------------------------------+------------+--+-
speecht5.decoder.prenet.embed_positions.weights      | UNEXPECTED |  | 
speecht5.encoder.prenet.pos_sinusoidal_embed.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ArTST v3 loaded! This model outputs diacritized text directly.
Parameters: 151,258,752


In [4]:
def load_audio(audio_path, sr=16000):
    try:
        audio, _ = librosa.load(audio_path, sr=sr)
        return audio
    except Exception as e:
        print(f"Audio load error: {e}")
        return None

@torch.inference_mode()
def transcribe_diacritized(audio):
    """Transcribe audio directly to diacritized Arabic text using ArTST v3."""
    if audio is None:
        return None
    
    inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_length=256)
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return transcription.strip()

def process_sample(text, audio_path):
    """Process sample - ArTST v3 outputs diacritized text directly."""
    if audio_path and Path(audio_path).exists():
        audio = load_audio(audio_path)
        if audio is not None:
            result = transcribe_diacritized(audio)
            if result:
                return result
    
    # If no audio, return undiacritized (this model needs audio)
    return text

In [5]:
# Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs, ref_pairs = extract_diacritics(pred_text), extract_diacritics(ref_text)
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Inference

In [6]:
# Checkpoint support for resume
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
print(f"Dev samples: {len(dev_input)}")

# Load checkpoint if exists
processed_ids, dev_predictions = load_checkpoint()
print(f"Resuming from checkpoint: {len(processed_ids)} already processed")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        diacritized = process_sample(item['text_undiacritized'], audio_path)
        dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error {item['id']}: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

Dev samples: 260
Resuming from checkpoint: 260 already processed


Dev Set: 100%|██████████| 260/260 [00:00<00:00, 1495910.89it/s]


In [7]:
print("="*60 + "\nDEV SET RESULTS\n" + "="*60)
m1 = compute_metrics(dev_predictions, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_predictions, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

DEV SET RESULTS

[With I'rab] DER: 78.76% | WER: 97.48% (PRIMARY) | SER: 99.62%
[No I'rab]   DER: 77.38% | WER: 97.48% | SER: 99.62%


## Blind Test Inference

In [8]:
# Checkpoint support for test set
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
print(f"Test samples: {len(test_input)}")

# Load checkpoint if exists
test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Resuming from checkpoint: {len(test_processed_ids)} already processed")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        diacritized = process_sample(item['text_undiacritized'], audio_path)
        test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")

Test samples: 328
Resuming from checkpoint: 328 already processed


Test Set: 100%|██████████| 328/328 [00:00<00:00, 1422680.16it/s]



SUBMISSION READY: ./submissions/artst_v3_diac_submission_20260222_054158.zip (14.4 KB)


In [9]:
# Google Drive sync removed for public release
